In [1]:
from google.colab import files
uploaded = files.upload()

Saving train_DUMM_dots_commas.csv to train_DUMM_dots_commas.csv


In [2]:
from google.colab import files
uploaded = files.upload()

Saving test_DUMM_dots_commas.csv to test_DUMM_dots_commas.csv


In [3]:
import os
os.listdir()

['.config',
 'test_DUMM_dots_commas.csv',
 'train_DUMM_dots_commas.csv',
 'sample_data']

In [4]:
# ============================================================
#  REDES NEURONALES (MLP) — Heating Load
#  Para ejecutar en Google Colab
#  Requiere: train_DUMM_dots_commas.csv y test_DUMM_dots_commas.csv
# ============================================================

import subprocess
subprocess.run(["pip", "install", "python-docx", "-q"])

import pandas as pd
import numpy as np
import time
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from docx import Document
from docx.shared import Inches, Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# ── Cargar datos ──────────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

print(f"Datos cargados — Train: {X_train.shape[0]} obs | Test: {X_test.shape[0]} obs")
print(f"Variables: {feature_names}")

# ── Estandarización (obligatoria para MLP) ────────────────────────────────────
# StandardScaler ajustado SOLO en train, aplicado a train y test
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# ── Funciones auxiliares ──────────────────────────────────────────────────────
def calcular_metricas(y_tr, yp_tr, y_te, yp_te):
    return {
        'r2_tr':   r2_score(y_tr, yp_tr),
        'r2_te':   r2_score(y_te, yp_te),
        'rmse_tr': np.sqrt(mean_squared_error(y_tr, yp_tr)),
        'rmse_te': np.sqrt(mean_squared_error(y_te, yp_te)),
        'mae_tr':  mean_absolute_error(y_tr, yp_tr),
        'mae_te':  mean_absolute_error(y_te, yp_te),
        'mape_te': np.mean(np.abs((y_te - yp_te) / y_te)) * 100,
    }

def imprimir_metricas(nombre, m):
    print(f"\n{'='*55}\n  {nombre}\n{'='*55}")
    print(f"  {'Métrica':<12} {'Train':>10} {'Test':>10}")
    print(f"  {'-'*32}")
    print(f"  {'R²':<12} {m['r2_tr']:>10.4f} {m['r2_te']:>10.4f}")
    print(f"  {'RMSE':<12} {m['rmse_tr']:>10.4f} {m['rmse_te']:>10.4f}")
    print(f"  {'MAE':<12} {m['mae_tr']:>10.4f} {m['mae_te']:>10.4f}")
    print(f"  {'MAPE (%)':<12} {'—':>10} {m['mape_te']:>9.2f}%")

def fig_to_buffer(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    return buf

# ── Helpers Word ──────────────────────────────────────────────────────────────
def añadir_parrafo(doc, texto):
    p = doc.add_paragraph(texto)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return p

def añadir_tabla(doc, cabeceras, filas, titulo=None):
    if titulo:
        p = doc.add_paragraph()
        run = p.add_run(titulo)
        run.bold = True
        run.font.size = Pt(10)
    tabla = doc.add_table(rows=1 + len(filas), cols=len(cabeceras))
    tabla.style = 'Table Grid'
    fila_cab = tabla.rows[0]
    for i, cab in enumerate(cabeceras):
        cell = fila_cab.cells[i]
        cell.text = cab
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        tc = cell._tc
        tcPr = tc.get_or_add_tcPr()
        shd = OxmlElement('w:shd')
        shd.set(qn('w:val'), 'clear')
        shd.set(qn('w:color'), 'auto')
        shd.set(qn('w:fill'), 'D9D9D9')
        tcPr.append(shd)
    for r, fila in enumerate(filas):
        for c, val in enumerate(fila):
            cell = tabla.rows[r + 1].cells[c]
            cell.text = str(val)
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()
    return tabla

def añadir_imagen(doc, buf, ancho=5.5, caption=None):
    doc.add_picture(buf, width=Inches(ancho))
    if caption:
        p = doc.add_paragraph(caption)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.runs[0].italic = True
        p.runs[0].font.size = Pt(9)
    doc.add_paragraph()

# ============================================================
#  MODELO BASE
# ============================================================
print("\n" + "="*55 + "\n  MODELO BASE: (64,32) ReLU Adam\n" + "="*55)

t0 = time.time()
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=2000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    tol=1e-4
)
mlp.fit(X_tr_sc, y_train)
t_mlp = time.time() - t0

yp_mlp_tr = mlp.predict(X_tr_sc)
yp_mlp_te = mlp.predict(X_te_sc)
met_mlp = calcular_metricas(y_train, yp_mlp_tr, y_test, yp_mlp_te)

print(f"Tiempo: {t_mlp:.2f} s  |  Iteraciones: {mlp.n_iter_}")
imprimir_metricas("Red Neuronal MLP — (64,32), ReLU, Adam, alpha=0.001", met_mlp)

# ============================================================
#  SENSIBILIDAD: ARQUITECTURA
# ============================================================
print("\n" + "="*55 + "\n  SENSIBILIDAD: ARQUITECTURA\n" + "="*55)

sens_arch = []
for arch in [(32,), (64,), (64, 32), (128, 64), (128, 64, 32), (64, 64)]:
    m = MLPRegressor(
        hidden_layer_sizes=arch, activation='relu', solver='adam',
        alpha=0.001, learning_rate_init=0.001, max_iter=2000,
        random_state=42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, tol=1e-4
    )
    m.fit(X_tr_sc, y_train)
    r2 = r2_score(y_test, m.predict(X_te_sc))
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_te_sc)))
    sens_arch.append((str(arch), round(r2, 4), round(rmse, 4)))
    print(f"  {str(arch):<20} → R²={r2:.4f}  RMSE={rmse:.4f}")

# ============================================================
#  SENSIBILIDAD: FUNCIÓN DE ACTIVACIÓN
# ============================================================
print("\n" + "="*55 + "\n  SENSIBILIDAD: FUNCIÓN DE ACTIVACIÓN\n" + "="*55)

sens_act = []
for act in ['relu', 'tanh', 'logistic']:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32), activation=act, solver='adam',
        alpha=0.001, learning_rate_init=0.001, max_iter=2000,
        random_state=42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, tol=1e-4
    )
    m.fit(X_tr_sc, y_train)
    r2 = r2_score(y_test, m.predict(X_te_sc))
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_te_sc)))
    sens_act.append((act, round(r2, 4), round(rmse, 4)))
    print(f"  {act:<10} → R²={r2:.4f}  RMSE={rmse:.4f}")

# ============================================================
#  SENSIBILIDAD: REGULARIZACIÓN (alpha)
# ============================================================
print("\n" + "="*55 + "\n  SENSIBILIDAD: REGULARIZACIÓN (alpha)\n" + "="*55)

sens_alpha = []
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
        alpha=alpha, learning_rate_init=0.001, max_iter=2000,
        random_state=42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, tol=1e-4
    )
    m.fit(X_tr_sc, y_train)
    r2 = r2_score(y_test, m.predict(X_te_sc))
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_te_sc)))
    sens_alpha.append((str(alpha), round(r2, 4), round(rmse, 4)))
    print(f"  alpha={alpha:<8} → R²={r2:.4f}  RMSE={rmse:.4f}")

# ============================================================
#  SENSIBILIDAD: LEARNING RATE
# ============================================================
print("\n" + "="*55 + "\n  SENSIBILIDAD: LEARNING RATE\n" + "="*55)

sens_lr = []
for lr in [0.01, 0.001, 0.0001]:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
        alpha=0.001, learning_rate_init=lr, max_iter=2000,
        random_state=42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, tol=1e-4
    )
    m.fit(X_tr_sc, y_train)
    r2 = r2_score(y_test, m.predict(X_te_sc))
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_te_sc)))
    sens_lr.append((str(lr), round(r2, 4), round(rmse, 4)))
    print(f"  lr={lr:<8} → R²={r2:.4f}  RMSE={rmse:.4f}")

# ============================================================
#  GRÁFICOS
# ============================================================

# Curva de convergencia
fig1, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp.loss_curve_, color='#2E75B6', label='Pérdida entrenamiento')
if hasattr(mlp, 'validation_scores_'):
    ax.plot([-v for v in mlp.validation_scores_], color='#ED7D31',
            linestyle='--', label='Pérdida validación')
ax.set_xlabel('Iteración')
ax.set_ylabel('Pérdida (MSE)')
ax.set_title('Red Neuronal — Curva de convergencia')
ax.legend()
plt.tight_layout()
buf_rn_loss = fig_to_buffer(fig1)
plt.close()

# Predichos vs reales
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_r, y_p, tit, col in zip(
    axes,
    [y_train, y_test],
    [yp_mlp_tr, yp_mlp_te],
    ['Train (N=576)', 'Test (N=192)'],
    ['#2E75B6', '#2E75B6']
):
    ax.scatter(y_r, y_p, alpha=0.5, s=20, color=col)
    mn = min(y_r.min(), y_p.min())
    mx = max(y_r.max(), y_p.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valores reales')
    ax.set_ylabel('Valores predichos')
    ax.set_title(f'Red Neuronal — {tit}')
    ax.legend()
plt.tight_layout()
buf_rn_pred = fig_to_buffer(fig2)
plt.close()

# ============================================================
#  DOCUMENTO WORD
# ============================================================
print("\nGenerando documento Word...")

doc = Document()
for section in doc.sections:
    section.top_margin    = Cm(2.5)
    section.bottom_margin = Cm(2.5)
    section.left_margin   = Cm(2.5)
    section.right_margin  = Cm(2.5)

# Título
doc.add_heading('4.5 Redes Neuronales (MLP)', level=1)

# Hiperparámetros modelo final
añadir_tabla(doc,
    ['Hiperparámetro', 'Valor', 'Descripción'],
    [
        ['Arquitectura',           '(64, 32)',  '2 capas ocultas: 64 y 32 neuronas'],
        ['Función de activación',  'ReLU',      'Rectified Linear Unit'],
        ['Optimizador',            'Adam',      'Adaptive Moment Estimation'],
        ['Regularización L2 (α)',  '0.001',     'Penalización sobre los pesos'],
        ['Learning rate inicial',  '0.001',     'Tasa de aprendizaje inicial'],
        ['Early stopping',         'Sí',        'Detención si no mejora en 20 épocas'],
        ['Fracción validación',    '10%',       'Del train, para early stopping'],
        ['Iteraciones',            str(mlp.n_iter_), 'Épocas hasta criterio de parada'],
        ['Tiempo ejecución',       f'{t_mlp:.2f} s', ''],
    ],
    titulo='Tabla RN-1. Hiperparámetros del modelo final — Red Neuronal MLP'
)

# Métricas
añadir_tabla(doc,
    ['Métrica', 'Train (N=576)', 'Test (N=192)'],
    [
        ['R²',      f"{met_mlp['r2_tr']:.4f}",   f"{met_mlp['r2_te']:.4f}"],
        ['RMSE',    f"{met_mlp['rmse_tr']:.4f}",  f"{met_mlp['rmse_te']:.4f}"],
        ['MAE',     f"{met_mlp['mae_tr']:.4f}",   f"{met_mlp['mae_te']:.4f}"],
        ['MAPE (%)', '—',                          f"{met_mlp['mape_te']:.2f}%"],
        ['Intervalos de predicción',  'No', 'No'],
        ['Coeficientes interpretables', 'No', 'No'],
    ],
    titulo='Tabla RN-2. Métricas Red Neuronal MLP'
)

# Sensibilidad arquitectura
añadir_tabla(doc,
    ['Arquitectura', 'R²_test', 'RMSE_test'],
    [(r[0], str(r[1]), str(r[2])) for r in sens_arch],
    titulo='Tabla RN-3. Sensibilidad a la arquitectura'
)

# Sensibilidad activación
añadir_tabla(doc,
    ['Función activación', 'R²_test', 'RMSE_test'],
    [(r[0], str(r[1]), str(r[2])) for r in sens_act],
    titulo='Tabla RN-4. Sensibilidad a la función de activación'
)

# Sensibilidad alpha
añadir_tabla(doc,
    ['Alpha (L2)', 'R²_test', 'RMSE_test'],
    [(r[0], str(r[1]), str(r[2])) for r in sens_alpha],
    titulo='Tabla RN-5. Sensibilidad a la regularización (alpha)'
)

# Sensibilidad learning rate
añadir_tabla(doc,
    ['Learning rate', 'R²_test', 'RMSE_test'],
    [(r[0], str(r[1]), str(r[2])) for r in sens_lr],
    titulo='Tabla RN-6. Sensibilidad al learning rate'
)

# Gráficos
añadir_imagen(doc, buf_rn_loss, ancho=5.0,
    caption='Figura RN-1. Curva de convergencia — Red Neuronal MLP')
añadir_imagen(doc, buf_rn_pred, ancho=5.8,
    caption='Figura RN-2. Valores predichos vs. reales — Red Neuronal MLP (train y test)')

# Guardar
nombre_doc = 'resultados_RN_heating_load.docx'
doc.save(nombre_doc)
print(f"\nDocumento Word guardado: {nombre_doc}")

try:
    from google.colab import files
    files.download(nombre_doc)
    print("Descarga iniciada.")
except ImportError:
    print(f"(No estás en Colab — archivo en directorio de trabajo)")

Datos cargados — Train: 576 obs | Test: 192 obs
Variables: ['relative_compactness', 'surface_area', 'wall_area', 'roof_area', 'overall_height', 'orientation_2', 'orientation_3', 'orientation_4', 'orientation_5', 'glazing_area_modified', 'glazing_dist_1', 'glazing_dist_2', 'glazing_dist_3', 'glazing_dist_4', 'glazing_dist_5', 'glazing_dist_6']

  MODELO BASE: (64,32) ReLU Adam
Tiempo: 1.73 s  |  Iteraciones: 339

  Red Neuronal MLP — (64,32), ReLU, Adam, alpha=0.001
  Métrica           Train       Test
  --------------------------------
  R²               0.9379     0.9062
  RMSE             2.5096     3.1032
  MAE              1.7250     2.1995
  MAPE (%)              —      9.67%

  SENSIBILIDAD: ARQUITECTURA
  (32,)                → R²=0.9224  RMSE=2.8223
  (64,)                → R²=0.8952  RMSE=3.2803
  (64, 32)             → R²=0.9062  RMSE=3.1032
  (128, 64)            → R²=0.9076  RMSE=3.0806
  (128, 64, 32)        → R²=0.9600  RMSE=2.0258
  (64, 64)             → R²=0.9239  RMSE

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada.


In [5]:
# ============================================================
#  REDES NEURONALES — SEGUNDA RONDA DE SENSIBILIDAD
#  Para ejecutar en Google Colab
#  Requiere: train_DUMM_dots_commas.csv y test_DUMM_dots_commas.csv
# ============================================================

import pandas as pd
import numpy as np
import time
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Cargar datos ──────────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

# ── Estandarización ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# ── Combinaciones a probar ────────────────────────────────────────────────────
combinaciones = [
    ((64, 32),       'tanh'),
    ((128, 64),      'tanh'),
    ((128, 64, 32),  'tanh'),
    ((64, 64),       'tanh'),
    ((128, 64, 32),  'relu'),
    ((128, 64, 32),  'logistic'),
]

print(f"\n{'='*65}")
print(f"  {'Arquitectura':<20} {'Activación':<12} {'R²_train':>9} {'R²_test':>9} {'RMSE_test':>10} {'RMSE_train':>11}")
print(f"  {'-'*63}")

resultados = []
for arch, act in combinaciones:
    m = MLPRegressor(
        hidden_layer_sizes=arch,
        activation=act,
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        tol=1e-4
    )
    m.fit(X_tr_sc, y_train)

    yp_tr = m.predict(X_tr_sc)
    yp_te = m.predict(X_te_sc)

    r2_tr   = r2_score(y_train, yp_tr)
    r2_te   = r2_score(y_test, yp_te)
    rmse_tr = np.sqrt(mean_squared_error(y_train, yp_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test, yp_te))
    mae_te  = mean_absolute_error(y_test, yp_te)
    mape_te = np.mean(np.abs((y_test - yp_te) / y_test)) * 100

    resultados.append({
        'arch': str(arch), 'act': act,
        'r2_tr': r2_tr, 'r2_te': r2_te,
        'rmse_tr': rmse_tr, 'rmse_te': rmse_te,
        'mae_te': mae_te, 'mape_te': mape_te,
        'iters': m.n_iter_
    })

    # Alerta sobreajuste: diferencia R² > 0.05
    alerta = " *** POSIBLE SOBREAJUSTE" if (r2_tr - r2_te) > 0.05 else ""
    print(f"  {str(arch):<20} {act:<12} {r2_tr:>9.4f} {r2_te:>9.4f} {rmse_te:>10.4f} {rmse_tr:>11.4f}{alerta}")

print(f"\n{'='*65}")
print("\nDetalle completo:")
print(f"\n  {'Arquitectura':<20} {'Activ.':<10} {'MAE_te':>8} {'MAPE_te':>9} {'Iters':>7}")
print(f"  {'-'*56}")
for r in resultados:
    print(f"  {r['arch']:<20} {r['act']:<10} {r['mae_te']:>8.4f} {r['mape_te']:>8.2f}% {r['iters']:>7}")

# Mejor modelo por RMSE_test
mejor = min(resultados, key=lambda x: x['rmse_te'])
print(f"\nMejor configuración por RMSE_test:")
print(f"  Arquitectura : {mejor['arch']}")
print(f"  Activación   : {mejor['act']}")
print(f"  R²_train     : {mejor['r2_tr']:.4f}")
print(f"  R²_test      : {mejor['r2_te']:.4f}")
print(f"  RMSE_test    : {mejor['rmse_te']:.4f}")
print(f"  MAE_test     : {mejor['mae_te']:.4f}")
print(f"  MAPE_test    : {mejor['mape_te']:.2f}%")
print(f"  Diferencia R²: {mejor['r2_tr'] - mejor['r2_te']:.4f} {'(OK)' if (mejor['r2_tr'] - mejor['r2_te']) <= 0.05 else '(SOBREAJUSTE)'}")


  Arquitectura         Activación    R²_train   R²_test  RMSE_test  RMSE_train
  ---------------------------------------------------------------
  (64, 32)             tanh            0.9967    0.9926     0.8692      0.5766
  (128, 64)            tanh            0.9917    0.9844     1.2673      0.9161
  (128, 64, 32)        tanh            0.9937    0.9903     1.0006      0.8006
  (64, 64)             tanh            0.8337    0.8380     4.0782      4.1053
  (128, 64, 32)        relu            0.9835    0.9600     2.0258      1.2948
  (128, 64, 32)        logistic        0.8336    0.8303     4.1749      4.1070


Detalle completo:

  Arquitectura         Activ.       MAE_te   MAPE_te   Iters
  --------------------------------------------------------
  (64, 32)             tanh         0.6183     3.19%    1425
  (128, 64)            tanh         0.8435     4.01%     810
  (128, 64, 32)        tanh         0.6377     3.24%     996
  (64, 64)             tanh         2.8197    11.40%    

In [6]:
# ============================================================
#  REDES NEURONALES — ROBUSTEZ CON DISTINTAS SEMILLAS
#  Para ejecutar en Google Colab
#  Requiere: train_DUMM_dots_commas.csv y test_DUMM_dots_commas.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Cargar datos ──────────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# ── Test de robustez ──────────────────────────────────────────────────────────
semillas = [0, 1, 7, 13, 42, 99, 123]

print(f"\n{'='*65}")
print(f"  Modelo: (64,32) tanh — test de robustez con {len(semillas)} semillas")
print(f"{'='*65}")
print(f"\n  {'Semilla':>8} {'R²_train':>10} {'R²_test':>10} {'RMSE_test':>11} {'Dif. R²':>9}")
print(f"  {'-'*50}")

resultados = []
for seed in semillas:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='tanh',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        tol=1e-4
    )
    m.fit(X_tr_sc, y_train)

    yp_tr = m.predict(X_tr_sc)
    yp_te = m.predict(X_te_sc)

    r2_tr   = r2_score(y_train, yp_tr)
    r2_te   = r2_score(y_test, yp_te)
    rmse_te = np.sqrt(mean_squared_error(y_test, yp_te))
    dif     = r2_tr - r2_te

    resultados.append({'seed': seed, 'r2_tr': r2_tr, 'r2_te': r2_te,
                       'rmse_te': rmse_te, 'dif': dif})

    alerta = " ***" if dif > 0.05 else ""
    print(f"  {seed:>8} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse_te:>11.4f} {dif:>9.4f}{alerta}")

# Estadísticos resumen
rmse_vals = [r['rmse_te'] for r in resultados]
r2_vals   = [r['r2_te']   for r in resultados]
dif_vals  = [r['dif']     for r in resultados]

print(f"\n{'='*65}")
print(f"  RESUMEN ESTADÍSTICO")
print(f"{'='*65}")
print(f"  RMSE_test — Media: {np.mean(rmse_vals):.4f}  |  Std: {np.std(rmse_vals):.4f}  |  Min: {np.min(rmse_vals):.4f}  |  Max: {np.max(rmse_vals):.4f}")
print(f"  R²_test   — Media: {np.mean(r2_vals):.4f}  |  Std: {np.std(r2_vals):.4f}  |  Min: {np.min(r2_vals):.4f}  |  Max: {np.max(r2_vals):.4f}")
print(f"  Dif. R²   — Media: {np.mean(dif_vals):.4f}  |  Std: {np.std(dif_vals):.4f}  |  Max: {np.max(dif_vals):.4f}")
print(f"\n  Semillas con posible sobreajuste (dif > 0.05): {sum(1 for d in dif_vals if d > 0.05)}/{len(semillas)}")


  Modelo: (64,32) tanh — test de robustez con 7 semillas

   Semilla   R²_train    R²_test   RMSE_test   Dif. R²
  --------------------------------------------------
         0     0.9762     0.9646      1.9071    0.0116
         1     0.9901     0.9815      1.3764    0.0086
         7     0.8510     0.8564      3.8403   -0.0054
        13     0.8645     0.8650      3.7227   -0.0006
        42     0.9967     0.9926      0.8692    0.0041
        99     0.8196     0.8215      4.2818   -0.0019
       123     0.9962     0.9912      0.9521    0.0050

  RESUMEN ESTADÍSTICO
  RMSE_test — Media: 2.4214  |  Std: 1.3676  |  Min: 0.8692  |  Max: 4.2818
  R²_test   — Media: 0.9247  |  Std: 0.0684  |  Min: 0.8215  |  Max: 0.9926
  Dif. R²   — Media: 0.0031  |  Std: 0.0056  |  Max: 0.0116

  Semillas con posible sobreajuste (dif > 0.05): 0/7


In [7]:
# ============================================================
#  REDES NEURONALES — ROBUSTEZ relu (128,64,32)
#  Para ejecutar en Google Colab
#  Requiere: train_DUMM_dots_commas.csv y test_DUMM_dots_commas.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Cargar datos ──────────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# ── Test de robustez ──────────────────────────────────────────────────────────
semillas = [0, 1, 7, 13, 42, 99, 123]

print(f"\n{'='*65}")
print(f"  Modelo: (128,64,32) relu — test de robustez con {len(semillas)} semillas")
print(f"{'='*65}")
print(f"\n  {'Semilla':>8} {'R²_train':>10} {'R²_test':>10} {'RMSE_test':>11} {'Dif. R²':>9}")
print(f"  {'-'*50}")

resultados = []
for seed in semillas:
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        tol=1e-4
    )
    m.fit(X_tr_sc, y_train)

    yp_tr = m.predict(X_tr_sc)
    yp_te = m.predict(X_te_sc)

    r2_tr   = r2_score(y_train, yp_tr)
    r2_te   = r2_score(y_test, yp_te)
    rmse_te = np.sqrt(mean_squared_error(y_test, yp_te))
    mae_te  = mean_absolute_error(y_test, yp_te)
    mape_te = np.mean(np.abs((y_test - yp_te) / y_test)) * 100
    dif     = r2_tr - r2_te

    resultados.append({
        'seed': seed, 'r2_tr': r2_tr, 'r2_te': r2_te,
        'rmse_te': rmse_te, 'mae_te': mae_te,
        'mape_te': mape_te, 'dif': dif
    })

    alerta = " ***" if dif > 0.05 else ""
    print(f"  {seed:>8} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse_te:>11.4f} {dif:>9.4f}{alerta}")

# Estadísticos resumen
rmse_vals = [r['rmse_te'] for r in resultados]
r2_vals   = [r['r2_te']   for r in resultados]
dif_vals  = [r['dif']     for r in resultados]

print(f"\n{'='*65}")
print(f"  RESUMEN ESTADÍSTICO")
print(f"{'='*65}")
print(f"  RMSE_test — Media: {np.mean(rmse_vals):.4f}  |  Std: {np.std(rmse_vals):.4f}  |  Min: {np.min(rmse_vals):.4f}  |  Max: {np.max(rmse_vals):.4f}")
print(f"  R²_test   — Media: {np.mean(r2_vals):.4f}  |  Std: {np.std(r2_vals):.4f}  |  Min: {np.min(r2_vals):.4f}  |  Max: {np.max(r2_vals):.4f}")
print(f"  Dif. R²   — Media: {np.mean(dif_vals):.4f}  |  Std: {np.std(dif_vals):.4f}  |  Max: {np.max(dif_vals):.4f}")
print(f"\n  Semillas con posible sobreajuste (dif > 0.05): {sum(1 for d in dif_vals if d > 0.05)}/{len(semillas)}")

# Mejor y peor semilla
mejor = min(resultados, key=lambda x: x['rmse_te'])
peor  = max(resultados, key=lambda x: x['rmse_te'])
print(f"\n  Mejor semilla: {mejor['seed']} — RMSE_test={mejor['rmse_te']:.4f}, R²_test={mejor['r2_te']:.4f}")
print(f"  Peor semilla:  {peor['seed']}  — RMSE_test={peor['rmse_te']:.4f}, R²_test={peor['r2_te']:.4f}")
print(f"  Rango RMSE:    {np.max(rmse_vals) - np.min(rmse_vals):.4f} (cuanto menor, más estable)")


  Modelo: (128,64,32) relu — test de robustez con 7 semillas

   Semilla   R²_train    R²_test   RMSE_test   Dif. R²
  --------------------------------------------------
         0     0.9511     0.9093      3.0512    0.0418
         1     0.9986     0.9943      0.7659    0.0043
         7     0.9352     0.9074      3.0831    0.0277
        13     0.9858     0.9675      1.8271    0.0183
        42     0.9835     0.9600      2.0258    0.0234
        99     0.9415     0.9058      3.1094    0.0356
       123     0.9288     0.8981      3.2342    0.0306

  RESUMEN ESTADÍSTICO
  RMSE_test — Media: 2.4424  |  Std: 0.8632  |  Min: 0.7659  |  Max: 3.2342
  R²_test   — Media: 0.9347  |  Std: 0.0355  |  Min: 0.8981  |  Max: 0.9943
  Dif. R²   — Media: 0.0260  |  Std: 0.0113  |  Max: 0.0418

  Semillas con posible sobreajuste (dif > 0.05): 0/7

  Mejor semilla: 1 — RMSE_test=0.7659, R²_test=0.9943
  Peor semilla:  123  — RMSE_test=3.2342, R²_test=0.8981
  Rango RMSE:    2.4683 (cuanto menor, más 

In [8]:
# ============================================================
#  REDES NEURONALES — ROBUSTEZ OPCIÓN A
#  Cambios respecto a ronda anterior:
#    1. early_stopping=False  → convergencia sin depender de
#       un subconjunto aleatorio de 52 observaciones
#    2. max_iter=5000, tol=1e-5  → criterio de parada más estricto
#    3. Tres arquitecturas simples en paralelo
#  Para ejecutar en Google Colab
#  Requiere: train_DUMM_dots_commas.csv y test_DUMM_dots_commas.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Cargar datos ──────────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# ── Configuración ─────────────────────────────────────────────────────────────
semillas     = [0, 1, 7, 13, 42, 99, 123]
arquitecturas = [
    ((64, 32),   'relu'),
    ((128, 64),  'relu'),
    ((64, 64),   'relu'),
]

# ── Bucle principal ───────────────────────────────────────────────────────────
todos = {}

for arch, act in arquitecturas:
    nombre = f"relu {arch}"
    print(f"\n{'='*65}")
    print(f"  Modelo: {nombre} — test de robustez con {len(semillas)} semillas")
    print(f"  [SIN early_stopping | max_iter=5000 | tol=1e-5]")
    print(f"{'='*65}")
    print(f"\n  {'Semilla':>8} {'R²_train':>10} {'R²_test':>10} {'RMSE_test':>11} {'Dif. R²':>9}")
    print(f"  {'-'*52}")

    resultados = []
    for seed in semillas:
        m = MLPRegressor(
            hidden_layer_sizes=arch,
            activation=act,
            solver='adam',
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=5000,
            tol=1e-5,
            early_stopping=False,   # ← CAMBIO CLAVE
            random_state=seed,
        )
        m.fit(X_tr_sc, y_train)

        yp_tr = m.predict(X_tr_sc)
        yp_te = m.predict(X_te_sc)

        r2_tr   = r2_score(y_train, yp_tr)
        r2_te   = r2_score(y_test,  yp_te)
        rmse_te = np.sqrt(mean_squared_error(y_test, yp_te))
        mae_te  = mean_absolute_error(y_test, yp_te)
        mape_te = np.mean(np.abs((y_test - yp_te) / y_test)) * 100
        dif     = r2_tr - r2_te

        resultados.append({
            'seed': seed, 'r2_tr': r2_tr, 'r2_te': r2_te,
            'rmse_te': rmse_te, 'mae_te': mae_te,
            'mape_te': mape_te, 'dif': dif,
            'iters': m.n_iter_
        })

        alerta = " ***" if dif > 0.05 else ""
        print(f"  {seed:>8} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse_te:>11.4f} {dif:>9.4f}{alerta}")

    # Estadísticos resumen por arquitectura
    rmse_vals = [r['rmse_te'] for r in resultados]
    r2_vals   = [r['r2_te']   for r in resultados]
    dif_vals  = [r['dif']     for r in resultados]

    print(f"\n  RESUMEN — {nombre}")
    print(f"  RMSE_test  Media={np.mean(rmse_vals):.4f}  Std={np.std(rmse_vals):.4f}  Min={np.min(rmse_vals):.4f}  Max={np.max(rmse_vals):.4f}  Rango={np.max(rmse_vals)-np.min(rmse_vals):.4f}")
    print(f"  R²_test    Media={np.mean(r2_vals):.4f}  Std={np.std(r2_vals):.4f}")
    print(f"  Dif. R²    Media={np.mean(dif_vals):.4f}  Max={np.max(dif_vals):.4f}")
    print(f"  Sobreajuste (dif>0.05): {sum(1 for d in dif_vals if d > 0.05)}/{len(semillas)}")

    mejor = min(resultados, key=lambda x: x['rmse_te'])
    peor  = max(resultados, key=lambda x: x['rmse_te'])
    print(f"  Mejor semilla: {mejor['seed']} — RMSE={mejor['rmse_te']:.4f}, R²={mejor['r2_te']:.4f}")
    print(f"  Peor  semilla: {peor['seed']}  — RMSE={peor['rmse_te']:.4f}, R²={peor['r2_te']:.4f}")

    todos[nombre] = {
        'resultados': resultados,
        'rmse_media': np.mean(rmse_vals),
        'rmse_std':   np.std(rmse_vals),
        'rmse_rango': np.max(rmse_vals) - np.min(rmse_vals),
        'r2_media':   np.mean(r2_vals),
    }

# ── Comparativa final entre arquitecturas ────────────────────────────────────
print(f"\n\n{'='*65}")
print(f"  COMPARATIVA FINAL ENTRE ARQUITECTURAS")
print(f"{'='*65}")
print(f"  {'Arquitectura':<20} {'RMSE_media':>11} {'RMSE_std':>10} {'RMSE_rango':>11} {'R²_media':>10}")
print(f"  {'-'*63}")
for nombre, stats in sorted(todos.items(), key=lambda x: x[1]['rmse_rango']):
    estab = "← MÁS ESTABLE" if nombre == min(todos, key=lambda x: todos[x]['rmse_rango']) else ""
    print(f"  {nombre:<20} {stats['rmse_media']:>11.4f} {stats['rmse_std']:>10.4f} {stats['rmse_rango']:>11.4f} {stats['r2_media']:>10.4f}  {estab}")

print(f"\n  Criterio de adopción:")
print(f"  → Rango RMSE < 0.5 y RMSE_media < 3.0  →  ADOPTAR como modelo final")
print(f"  → Rango RMSE ≥ 0.5  →  pasar a Opción B/C")


  Modelo: relu (64, 32) — test de robustez con 7 semillas
  [SIN early_stopping | max_iter=5000 | tol=1e-5]

   Semilla   R²_train    R²_test   RMSE_test   Dif. R²
  ----------------------------------------------------
         0     0.9997     0.9938      0.7961    0.0059
         1     0.9997     0.9969      0.5656    0.0028
         7     0.9993     0.9901      1.0105    0.0093
        13     0.9995     0.9935      0.8152    0.0060
        42     0.9995     0.9915      0.9362    0.0080
        99     0.9994     0.9939      0.7935    0.0055
       123     0.9994     0.9886      1.0827    0.0108

  RESUMEN — relu (64, 32)
  RMSE_test  Media=0.8571  Std=0.1582  Min=0.5656  Max=1.0827  Rango=0.5171
  R²_test    Media=0.9926  Std=0.0026
  Dif. R²    Media=0.0069  Max=0.0108
  Sobreajuste (dif>0.05): 0/7
  Mejor semilla: 1 — RMSE=0.5656, R²=0.9969
  Peor  semilla: 123  — RMSE=1.0827, R²=0.9886

  Modelo: relu (128, 64) — test de robustez con 7 semillas
  [SIN early_stopping | max_iter=50

In [9]:
# ============================================================
#  4.5 REDES NEURONALES — Trabajo Final Econometría
#  Base: Tsanas & Xifara (2012) — Heating Load
#  Modelo definitivo: relu (64,32), early_stopping=False
#  Métricas: media de 7 semillas
#  Gráficos: mismo estilo que RF (scatter azul #2E75B6)
#  IMPORTANTE: subir train_DUMM_dots_commas.csv y
#              test_DUMM_dots_commas.csv antes de ejecutar
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── 1. CARGAR DATOS ───────────────────────────────────────────────────────────
train = pd.read_csv('train_DUMM_dots_commas.csv')
test  = pd.read_csv('test_DUMM_dots_commas.csv')
train.columns = train.columns.str.replace('\ufeff', '')
test.columns  = test.columns.str.replace('\ufeff', '')

TARGET = 'heating_load'
feature_names = [c for c in train.columns if c != TARGET]
y_train = train[TARGET].values
X_train = train[feature_names].values
y_test  = test[TARGET].values
X_test  = test[feature_names].values

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f"✓ Datos cargados")
print(f"  Train: {X_train.shape[0]} obs | {X_train.shape[1]} variables")
print(f"  Test:  {X_test.shape[0]} obs")

# ── 2. TEST DE ROBUSTEZ — 7 SEMILLAS ─────────────────────────────────────────
semillas = [0, 1, 7, 13, 42, 99, 123]

print("\n" + "="*65)
print("  MODELO: relu (64,32) — 7 semillas")
print("  [early_stopping=False | max_iter=5000 | tol=1e-5]")
print("="*65)
print(f"\n  {'Semilla':>8} {'R²_train':>10} {'R²_test':>10} {'RMSE_test':>11} "
      f"{'MAE_test':>10} {'MAPE_test':>10} {'Dif. R²':>9}")
print(f"  {'-'*70}")

resultados = []
# Guardamos predicciones de la semilla con RMSE más cercano a la media
# (la usaremos para el gráfico; se determina tras el bucle)
preds_por_semilla = {}

for seed in semillas:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=5000,
        tol=1e-5,
        early_stopping=False,
        random_state=seed,
    )
    m.fit(X_tr_sc, y_train)

    yp_tr = m.predict(X_tr_sc)
    yp_te = m.predict(X_te_sc)

    r2_tr   = r2_score(y_train, yp_tr)
    r2_te   = r2_score(y_test,  yp_te)
    rmse_te = np.sqrt(mean_squared_error(y_test, yp_te))
    mae_te  = mean_absolute_error(y_test, yp_te)
    mape_te = np.mean(np.abs((y_test - yp_te) / y_test)) * 100
    dif     = r2_tr - r2_te

    resultados.append({
        'seed': seed, 'r2_tr': r2_tr, 'r2_te': r2_te,
        'rmse_te': rmse_te, 'mae_te': mae_te,
        'mape_te': mape_te, 'dif': dif,
        'yp_tr': yp_tr, 'yp_te': yp_te
    })

    print(f"  {seed:>8} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse_te:>11.4f} "
          f"{mae_te:>10.4f} {mape_te:>9.2f}% {dif:>9.4f}")

# ── 3. MÉTRICAS MEDIAS ────────────────────────────────────────────────────────
rmse_vals = [r['rmse_te'] for r in resultados]
r2_vals   = [r['r2_te']   for r in resultados]
mae_vals  = [r['mae_te']  for r in resultados]
mape_vals = [r['mape_te'] for r in resultados]
dif_vals  = [r['dif']     for r in resultados]

rmse_media = np.mean(rmse_vals)
r2_media   = np.mean(r2_vals)
mae_media  = np.mean(mae_vals)
mape_media = np.mean(mape_vals)

print(f"\n{'='*65}")
print(f"  MÉTRICAS MEDIAS (7 semillas)")
print(f"{'='*65}")
print(f"  R²_test    : {r2_media:.4f}  (std={np.std(r2_vals):.4f})")
print(f"  RMSE_test  : {rmse_media:.4f}  (std={np.std(rmse_vals):.4f})")
print(f"  MAE_test   : {mae_media:.4f}  (std={np.std(mae_vals):.4f})")
print(f"  MAPE_test  : {mape_media:.2f}%  (std={np.std(mape_vals):.2f}%)")
print(f"  Dif. R²    : {np.mean(dif_vals):.4f}  (max={np.max(dif_vals):.4f})")
print(f"  Rango RMSE : {np.max(rmse_vals)-np.min(rmse_vals):.4f}")
print(f"  Sobreajuste (dif>0.05): {sum(1 for d in dif_vals if d > 0.05)}/7")

mejor  = min(resultados, key=lambda x: x['rmse_te'])
mediana_seed = min(resultados, key=lambda x: abs(x['rmse_te'] - rmse_media))

print(f"\n  Mejor semilla : {mejor['seed']} — RMSE={mejor['rmse_te']:.4f}")
print(f"  Semilla usada para gráficos: {mediana_seed['seed']} "
      f"(RMSE={mediana_seed['rmse_te']:.4f}, más cercana a la media)")

# ── 4. GRÁFICOS ───────────────────────────────────────────────────────────────
# Usamos la semilla cuyo RMSE es más cercano a la media → representativa
yp_tr_graf = mediana_seed['yp_tr']
yp_te_graf = mediana_seed['yp_te']

# ── Gráfico 1: Predichos vs Reales (mismo estilo que RF) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_r, y_p, tit in zip(
    axes,
    [y_train, y_test],
    [yp_tr_graf, yp_te_graf],
    [f'Train (N=576)', f'Test (N=192)']
):
    ax.scatter(y_r, y_p, alpha=0.5, s=20, color='#2E75B6')
    mn = min(y_r.min(), y_p.min())
    mx = max(y_r.max(), y_p.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valores reales (kWh/m²)')
    ax.set_ylabel('Valores predichos (kWh/m²)')
    ax.set_title(f'Red Neuronal relu(64,32) — {tit}')
    ax.legend()

plt.tight_layout()
plt.savefig('RN_predichos_vs_reales.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Guardado: RN_predichos_vs_reales.png")

# ── Gráfico 2: Estabilidad entre semillas (RMSE por semilla) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

seeds_labels = [str(s) for s in semillas]

# RMSE por semilla
axes[0].bar(seeds_labels, rmse_vals, color='#2E75B6', alpha=0.8, edgecolor='white')
axes[0].axhline(rmse_media, color='red', lw=1.5, linestyle='--',
                label=f'Media = {rmse_media:.3f}')
axes[0].set_xlabel('Semilla')
axes[0].set_ylabel('RMSE_test (kWh/m²)')
axes[0].set_title('RMSE_test por semilla')
axes[0].legend()
axes[0].set_ylim(0, max(rmse_vals) * 1.15)

# R² por semilla
axes[1].bar(seeds_labels, r2_vals, color='#2E75B6', alpha=0.8, edgecolor='white')
axes[1].axhline(r2_media, color='red', lw=1.5, linestyle='--',
                label=f'Media = {r2_media:.4f}')
axes[1].set_xlabel('Semilla')
axes[1].set_ylabel('R²_test')
axes[1].set_title('R²_test por semilla')
axes[1].legend()
axes[1].set_ylim(min(r2_vals) * 0.997, 1.0)

plt.suptitle('Red Neuronal relu(64,32) — Estabilidad entre semillas', fontsize=12)
plt.tight_layout()
plt.savefig('RN_estabilidad_semillas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Guardado: RN_estabilidad_semillas.png")

# ── Gráfico 3: Residuos (mismo estilo que RF) ─────────────────────────────────
resid_te = y_test - yp_te_graf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(yp_te_graf, resid_te, alpha=0.5, s=20, color='#2E75B6')
axes[0].axhline(0, color='red', lw=1, linestyle='--')
axes[0].set_xlabel('Valores predichos')
axes[0].set_ylabel('Residuo')
axes[0].set_title('Residuos vs. Valores predichos (test)')

axes[1].hist(resid_te, bins=25, color='#2E75B6', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Histograma de residuos (test)')

plt.tight_layout()
plt.savefig('RN_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Guardado: RN_residuos.png")

# ── 5. RESUMEN FINAL ──────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  ✅ RED NEURONAL COMPLETADA")
print("="*65)
print(f"  Arquitectura : relu (64, 32)")
print(f"  R²_test      : {r2_media:.4f}  (media 7 semillas)")
print(f"  RMSE_test    : {rmse_media:.4f}  (media 7 semillas)")
print(f"  MAE_test     : {mae_media:.4f}  (media 7 semillas)")
print(f"  MAPE_test    : {mape_media:.2f}%  (media 7 semillas)")
print(f"  Rango RMSE   : {np.max(rmse_vals)-np.min(rmse_vals):.4f}")
print("="*65)
print("\nArchivos generados:")
print("  - RN_predichos_vs_reales.png")
print("  - RN_estabilidad_semillas.png")
print("  - RN_residuos.png")
print("\nMétricas para la tabla del Word:")
print(f"  RMSE_test = {rmse_media:.3f}*")
print(f"  MAE_test  = {mae_media:.3f}*")
print(f"  MAPE_test = {mape_media:.2f}%*")
print(f"  R²_test   = {r2_media:.4f}*")
print("  (* media de 7 semillas)")

✓ Datos cargados
  Train: 576 obs | 16 variables
  Test:  192 obs

  MODELO: relu (64,32) — 7 semillas
  [early_stopping=False | max_iter=5000 | tol=1e-5]

   Semilla   R²_train    R²_test   RMSE_test   MAE_test  MAPE_test   Dif. R²
  ----------------------------------------------------------------------
         0     0.9997     0.9938      0.7961     0.5683      3.19%    0.0059
         1     0.9997     0.9969      0.5656     0.4079      2.42%    0.0028
         7     0.9993     0.9901      1.0105     0.7181      4.00%    0.0093
        13     0.9995     0.9935      0.8152     0.5531      3.03%    0.0060
        42     0.9995     0.9915      0.9362     0.6218      3.85%    0.0080
        99     0.9994     0.9939      0.7935     0.5627      3.19%    0.0055
       123     0.9994     0.9886      1.0827     0.6828      4.80%    0.0108

  MÉTRICAS MEDIAS (7 semillas)
  R²_test    : 0.9926  (std=0.0026)
  RMSE_test  : 0.8571  (std=0.1582)
  MAE_test   : 0.5878  (std=0.0939)
  MAPE_test  : 